# Wyznaczanie orbity z 3 obserwacji: pełny przykład w PythonieTen notebook pokazuje, jak z trzech obserwacji astrometrycznych (RA/Dec, czasy) uzyskać wstępną orbitę nowego obiektu oraz ją rafinować.Co robimy:- Generujemy syntetyczne dane obserwacyjne z zadanych elementów orbity ("prawda") oraz prostego modelu Ziemi (orbita kołowa 1 AU w płaszczyźnie ekliptyki, z przejściem do układu równikowego).- Szacujemy orbitę metodą prostego "ranging": przeszukujemy siatkę odległości i prędkości radialnej w chwili środkowej, budujemy stan i propagujemy go metodą dwuciałową, minimalizując błąd kątowy.- Rafinujemy wynik metodą najmniejszych kwadratów (differential correction).- Porównujemy elementy Keplerowskie odzyskane i prawdziwe.Uproszczenia: brak opóźnienia światła, topocentrum pominięte, Ziemia po kole. To demonstracja edukacyjna.

In [ ]:
import math, numpy as npfrom dataclasses import dataclassfrom typing import Tuplefrom scipy.optimize import minimize, least_squaresnp.set_printoptions(precision=6, suppress=True)K_GAUSS = 0.01720209895MU_SUN = K_GAUSS**2EPS = math.radians(23.439291111)

In [ ]:
def norm(v): return np.linalg.norm(v)def unit(v): return v/norm(v)def rot_x(a): c,s=math.cos(a),math.sin(a); return np.array([[1,0,0],[0,c,-s],[0,s,c]],float)def rot_z(a): c,s=math.cos(a),math.sin(a); return np.array([[c,-s,0],[s,c,0],[0,0,1]],float)R_ECL_TO_EQU = rot_x(EPS)def ra_dec_from_vec(r):    x,y,z = r; rr = math.sqrt(x*x+y*y+z*z)    return (math.atan2(y,x)%(2*math.pi), math.asin(z/rr))def uvec_from_ra_dec(ra,dec):    return np.array([math.cos(dec)*math.cos(ra), math.cos(dec)*math.sin(ra), math.sin(dec)])

In [ ]:
# Stumpff + uni. variables propagationdef C(z):    if z>1e-8: s=math.sqrt(z); return (1-math.cos(s))/z    if z<-1e-8: s=math.sqrt(-z); return (math.cosh(s)-1)/(-z)    return 0.5 - z/24 + z*z/720 - z*z*z/40320def S(z):    if z>1e-8: s=math.sqrt(z); return (s-math.sin(s))/s**3    if z<-1e-8: s=math.sqrt(-z); return (math.sinh(s)-s)/s**3    return 1/6 - z/120 + z*z/5040 - z*z*z/362880def propagate_universal(r0,v0,dt,mu=MU_SUN):    r0n=norm(r0); a=2/r0n - np.dot(v0,v0)/mu    chi=math.sqrt(mu)*abs(a)*dt if abs(a)>1e-12 else 1.0    for _ in range(100):        z=a*chi*chi; c=C(z); s=S(z)        r = r0n*c*chi*chi + (np.dot(r0,v0)/math.sqrt(mu))*s*chi**3 + r0n*chi        F = r - math.sqrt(mu)*dt        if abs(F)<1e-12: break        dF = r0n*(1 - z*s) + (np.dot(r0,v0)/math.sqrt(mu))*(1 - z*c) + (1 - 0.5*z)*chi        chi -= F/dF    z=a*chi*chi; c=C(z); s=S(z)    f = 1 - (chi*chi/r0n)*c    g = dt - (1/math.sqrt(mu))*chi**3*s    r = f*r0 + g*v0; rn=norm(r)    fd = (math.sqrt(mu)/(rn*r0n))*(z*s - 1)*chi    gd = 1 - (chi*chi/rn)*c    v = fd*r0 + gd*v0    return r, v

In [ ]:
def coe_from_rv(r,v,mu=MU_SUN):    R=norm(r); V=norm(v); h=np.cross(r,v); H=norm(h)    i=math.acos(h[2]/H); K=np.array([0,0,1.0]); n=np.cross(K,h); N=norm(n)    e_vec=(1/mu)*((V*V-mu/R)*r - np.dot(r,v)*v); e=norm(e_vec)    E=V*V/2 - mu/R; a=-mu/(2*E) if abs(e-1)>1e-12 else math.inf    Omega=0.0 if N<1e-12 else math.acos(np.clip(n[0]/N,-1,1)); Omega = 2*math.pi - Omega if (N>=1e-12 and n[1]<0) else Omega    omega=0.0 if (N<1e-12 or e<1e-12) else math.acos(np.clip(np.dot(n,e_vec)/(N*e),-1,1)); omega = 2*math.pi - omega if (e>=1e-12 and e_vec[2]<0) else omega    if e>1e-12:        nu=math.acos(np.clip(np.dot(e_vec,r)/(e*R),-1,1)); nu=2*math.pi - nu if np.dot(r,v)<0 else nu    else: nu=0.0    return a,e,i,Omega,omega,nudef rv_from_coe(a,e,inc,Omega,omega,nu,mu=MU_SUN):    def rot_x(a): c,s=math.cos(a),math.sin(a); return np.array([[1,0,0],[0,c,-s],[0,s,c]])    def rot_z(a): c,s=math.cos(a),math.sin(a); return np.array([[c,-s,0],[s,c,0],[0,0,1]])    p=a*(1-e*e)    r_pqw=np.array([p*math.cos(nu)/(1+e*math.cos(nu)), p*math.sin(nu)/(1+e*math.cos(nu)), 0.0])    v_pqw=math.sqrt(mu/p)*np.array([-math.sin(nu), e+math.cos(nu), 0.0])    Q=rot_z(Omega)@rot_x(inc)@rot_z(omega)    return Q@r_pqw, Q@v_pqw

In [ ]:
def earth_heliocentric_equatorial(t_days,L0=0.0,R_AU=1.0):    n=K_GAUSS; L=L0+n*t_days    r_ecl=np.array([R_AU*math.cos(L),R_AU*math.sin(L),0.0])    v_ecl=n*np.array([-R_AU*math.sin(L),R_AU*math.cos(L),0.0])    return R_ECL_TO_EQU@r_ecl, R_ECL_TO_EQU@v_ecl@dataclassclass Obs: t: float; ra: float; dec: floatdef make_synthetic_case():    a_true=2.45; e_true=0.17; i_true=math.radians(10.0); Omega_true=math.radians(80.0); omega_true=math.radians(250.0); M0_true=math.radians(20.0)    def solve_kepler(M,e):        E=M        for _ in range(50):            f=E-e*math.sin(E)-M; fp=1-e*math.cos(E); dE=-f/fp; E+=dE            if abs(dE)<1e-12: break        nu=math.atan2(math.sqrt(1-e*e)*math.sin(E), math.cos(E)-e); return E, (nu%(2*math.pi))    t2=0.0; t1=-7.0; t3=7.0; n_obj=math.sqrt(MU_SUN/(a_true**3))    M1=(M0_true - n_obj*(t2-t1))%(2*math.pi); M2=M0_true%(2*math.pi); M3=(M0_true + n_obj*(t3-t2))%(2*math.pi)    _,nu1=solve_kepler(M1,e_true); _,nu2=solve_kepler(M2,e_true); _,nu3=solve_kepler(M3,e_true)    r1,v1=rv_from_coe(a_true,e_true,i_true,Omega_true,omega_true,nu1)    r2,v2=rv_from_coe(a_true,e_true,i_true,Omega_true,omega_true,nu2)    r3,v3=rv_from_coe(a_true,e_true,i_true,Omega_true,omega_true,nu3)    R1,_=earth_heliocentric_equatorial(t1); R2,_=earth_heliocentric_equatorial(t2); R3,_=earth_heliocentric_equatorial(t3)    u1=unit(r1-R1); u2=unit(r2-R2); u3=unit(r3-R3)    ra1,dec1=ra_dec_from_vec(u1); ra2,dec2=ra_dec_from_vec(u2); ra3,dec3=ra_dec_from_vec(u3)    obs=[Obs(t1,ra1,dec1),Obs(t2,ra2,dec2),Obs(t3,ra3,dec3)]    truth={'a':a_true,'e':e_true,'i':i_true,'Omega':Omega_true,'omega':omega_true,'M0_at_t2':M0_true,'r_true':(r1,r2,r3),'v_true':(v1,v2,v3),'R_earth':(R1,R2,R3)}    return obs, truthobs, truth = make_synthetic_case()for k,o in enumerate(obs,1): print(f"O{k}: t={o.t:+.1f} d, RA={math.degrees(o.ra)/15:.6f} h, Dec={math.degrees(o.dec):+.6f} deg")

In [ ]:
def ang_sep(u,v):    c=np.clip(np.dot(unit(u),unit(v)),-1,1); return math.acos(c)def ranging_initial_orbit(obs, truth):    (R1,R2,R3)=truth['R_earth']    o1,o2,o3=obs    u1=uvec_from_ra_dec(o1.ra,o1.dec)    u2=uvec_from_ra_dec(o2.ra,o2.dec)    u3=uvec_from_ra_dec(o3.ra,o3.dec)    dt=o3.t-o1.t    u_dot=(u3-u1)/dt    u_dot_perp=u_dot - np.dot(u_dot,u2)*u2    def cost(x):        rho2,rhod2=x        r2=R2 + rho2*u2        v2 = rhod2*u2 + rho2*u_dot_perp        r1p,_=propagate_universal(r2,v2,o1.t-o2.t); r3p,_=propagate_universal(r2,v2,o3.t-o2.t)        a1=ang_sep(unit(r1p-R1),u1); a3=ang_sep(unit(r3p-R3),u3); return a1*a1+a3*a3    grid_rho=np.linspace(0.5,4.0,50); grid_rhod=np.linspace(-0.01,0.01,50); best=None; x0=None    for rr in grid_rho:        for vv in grid_rhod:            val=cost((rr,vv))            if best is None or val<best: best=val; x0=(rr,vv)    res=minimize(lambda x: cost(x), x0=np.array(x0), method='Nelder-Mead', options={'maxiter':2000})    rho2,rhod2=res.x    r2=R2 + rho2*u2; v2 = rhod2*u2 + rho2*u_dot_perp    return r2, v2, res.successr2_i,v2_i,ok = ranging_initial_orbit(obs, truth)print('Ranging OK:', ok)a,e,i,O,w,nu = coe_from_rv(r2_i, v2_i)print(f"Start: a={a:.6f} AU e={e:.6f} i={math.degrees(i):.3f} Ω={math.degrees(O):.3f} ω={math.degrees(w):.3f}")

In [ ]:
def residuals_state6(x, obs, truth):    (R1,R2,R3)=truth['R_earth']    o1,o2,o3=obs    r2=x[:3]; v2=x[3:]    res=[]    for o,R,dt in [(o1,R1,o1.t-o2.t),(o2,R2,0.0),(o3,R3,o3.t-o2.t)]:        rp,_=propagate_universal(r2,v2,dt)        u_pred=unit(rp-R); u_obs=uvec_from_ra_dec(o.ra,o.dec)        ra,dec=ra_dec_from_vec(u_obs)        e_ra=np.array([-math.sin(ra), math.cos(ra), 0.0])*math.cos(dec)        e_dec=np.array([-math.cos(ra)*math.sin(dec), -math.sin(ra)*math.sin(dec), math.cos(dec)])        d=u_pred-u_obs        res.append(np.dot(d,e_ra)); res.append(np.dot(d,e_dec))    return np.array(res)x0=np.hstack([r2_i,v2_i])ls=least_squares(lambda x: residuals_state6(x,obs,truth), x0, method='lm', max_nfev=200)r2_f=ls.x[:3]; v2_f=ls.x[3:]a,e,i,O,w,nu = coe_from_rv(r2_f, v2_f)print('LSQ success:', ls.success)print(f"Fit: a={a:.6f} AU e={e:.6f} i={math.degrees(i):.3f} Ω={math.degrees(O):.3f} ω={math.degrees(w):.3f}")